# Data Analysis · Week 16, session 1 of 2
## Visualisation and matplotlib

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

A chart is an argument. Everything in this session exists so that argument can be read by
somebody who was not in the room when you made it.

By the end of this notebook you will be able to:

1. Pick the chart from the question: bar, line, scatter or histogram.
2. Build a chart with `matplotlib`, using a figure and axes, and save it as an image.
3. Title the chart with the finding rather than with the names of the axes.
4. Format the axes so nobody has to count digits.
5. Choose accessible colour, and never let colour be the only signal.

### How to use this notebook

Run the cells in order. The charts appear below the cell that draws them, so you see the
effect of every change straight away.

Two cells draw a deliberately bad chart so you can compare. They carry a comment saying so.

---
## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)

In [ ]:
# Plumbing, not the lesson. This cell puts the three course CSVs within
# reach of pandas, and it never needs running again.
#
# It looks for them in the repository first, which is public and reads
# over a URL. If that does not answer, it rebuilds them right here from
# the course's fixed seed, so both routes produce identical files.
# Nothing ever has to be uploaded by hand.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
FILES = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in FILES:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Write the three CSV files again from the course's fixed seed.

    They come out byte for byte identical to the ones in the repository,
    so the numbers on the slide still match the ones in the notebook.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # the deliberate dirt: one region typed four ways, blank cells, and
    # rows captured twice
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Data read from the repository.")
except Exception:
    _reconstruir_datos()
    print("The repository did not answer. Data rebuilt in this session.")

print("Ready:", ", ".join(FILES))

In [ ]:
# The cleaning from session 15.2, in one cell, so this notebook opens on its own.
sales = pd.read_csv("sales.csv").drop_duplicates()
sales["region"] = sales["region"].str.strip().str.title()
sales["unit_price"] = (sales["unit_price"]
                       .str.replace("$", "", regex=False)
                       .str.replace(",", "", regex=False)
                       .str.strip().astype(float))
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.dropna(subset=["units"])
sales["units"] = sales["units"].astype(int)
sales["amount"] = sales["units"] * sales["unit_price"]

employees = pd.read_csv("employees.csv")
monthly = sales.groupby(sales["date"].dt.month)["amount"].sum()

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

print(f"{len(sales)} clean rows, {len(employees)} employees")

---
# Block 1 · Which chart

This is not a style decision. Each shape answers one kind of question, and using the wrong one
makes a true number say something false.

| Chart | The question it answers | Example from the course |
|---|---|---|
| Bar | How do these categories compare? | Revenue per product |
| Line | How did this change over time? | Revenue per month |
| Scatter | Do these two numbers move together? | Salary against tenure |
| Histogram | How are the values spread out? | Distribution of salaries |

All four, drawn with the course data, one at a time.

## Bar: comparing categories

Sorted, because an unsorted bar chart makes the reader do the ranking by eye. Horizontal,
because the category names are words and words read across.

In [ ]:
by_product = sales.groupby("product")["amount"].sum().sort_values() / 1000

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(by_product.index, by_product.values, color="#2B5F8F")
ax.set_title("Which product brings the most revenue?", loc="left", fontweight="bold")
ax.set_xlabel("Thousands of pesos")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

The espresso machine brings more than half the revenue, and the order of the bars has already
answered the question without anyone having to compare lengths.

## Line: change along an ordered axis

A line tells the reader the points are connected in an order that means something. That is
true between January and February.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(MONTHS, monthly.values / 1000, marker="o", linewidth=2, color="#2B5F8F")
ax.set_title("How did revenue move through the year?", loc="left", fontweight="bold")
ax.set_ylabel("Thousands of pesos")
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

### The rule that follows from that

Connecting two points with a line asserts there is a journey between them. Between January and
February that is true. Between North and South it is false, and the reader will believe it
because the shape is telling them so.

In [ ]:
# DRAWS BADLY ON PURPOSE. A line across categories invents a journey.
by_region = sales.groupby("region")["amount"].sum() / 1000

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(by_region.index, by_region.values, marker="o", color="#B4530A", linewidth=2)
axes[0].set_title("Wrong: does North lead to Centre?", loc="left", fontweight="bold")

axes[1].bar(by_region.index, by_region.values, color="#2B5F8F")
axes[1].set_title("Right: four comparable things", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Thousands of pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

Both charts carry exactly the same four numbers. The left one suggests the regions sit in a
sequence and that there is a fall from Centre to South, when the order is alphabetical and
means nothing.

## Scatter: the relationship between two numbers

One dot per row, positioned by two of its values. It is the chart that answers whether more of
this comes with more of that.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(employees["tenure_months"], employees["monthly_salary"],
           alpha=0.55, color="#2B5F8F", edgecolor="none")
ax.set_title("Does salary rise with tenure?", loc="left", fontweight="bold")
ax.set_xlabel("Tenure in months")
ax.set_ylabel("Monthly salary")

r = employees["tenure_months"].corr(employees["monthly_salary"])
ax.annotate(f"correlation = {r:.2f}", xy=(0.04, 0.92), xycoords="axes fraction",
            fontsize=10, color="#5B6B84")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

print("Correlation:", round(r, 3))

The correlation puts a number on what the eye is doing. It runs from minus one to one.

Here it comes out at 0.28, a weak relationship: the cloud drifts up to the right and there are
still people with two years earning more than people with ten. A number near zero means the
cloud has no direction, and a strong number **still does not mean** one caused the other.

## Histogram: how one column is spread out

A histogram slices one column into ranges and counts how many rows land in each. It answers
what typical looks like and how wide the spread is.

A bar chart compares named things; a histogram compares ranges of one thing. It is the
distinction people confuse most out of the four.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(employees["monthly_salary"], bins=15, color="#2B5F8F", edgecolor="white")
ax.set_title("How are salaries spread out?", loc="left", fontweight="bold")
ax.set_xlabel("Monthly salary")
ax.set_ylabel("Employees")

mean_salary = employees["monthly_salary"].mean()
ax.axvline(mean_salary, color="#B4530A", linestyle="--", linewidth=2)
ax.annotate(f"mean {mean_salary:,.0f}", xy=(mean_salary, 0), xytext=(6, 6),
            textcoords="offset points", color="#B4530A", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

print("Mean:  ", round(mean_salary))
print("Median:", employees["monthly_salary"].median())

The mean drawn on top shows how much it hides. Most people earn below it, and a handful of
high salaries pull it to the right. Reporting only the mean of this column would give a wrong
idea of what a typical person earns.

## The one that almost never works

A pie chart asks the reader to compare angles, which people do badly. Past three slices it
stops being readable. Draw both with the same numbers and the difference shows itself.

In [ ]:
# DRAWS BADLY ON PURPOSE, on the left. Both panels carry the same data.
shares = sales.groupby("product")["amount"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].pie(shares.values, labels=shares.index, autopct="%1.0f%%",
            startangle=90, colors=plt.cm.Blues(range(60, 260, 40)))
axes[0].set_title("As a pie: which two are closest?", loc="left", fontweight="bold")

axes[1].barh(shares.sort_values().index, shares.sort_values().values / 1000, color="#2B5F8F")
axes[1].set_title("As bars: now you can tell", loc="left", fontweight="bold")
axes[1].set_xlabel("Thousands of pesos")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.tight_layout()
plt.show()

---
# Block 2 · How a chart is built

Two objects, and every matplotlib chart starts with the same line.

A **figure** is the sheet of paper. An **axes** is one set of axes drawn on it. `subplots()`
hands you both at once, and that is how practically every chart you write will begin.

You draw on the axes, and you save the figure.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(MONTHS, monthly.values / 1000)

fig.savefig("first.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Wrote first.png")

`dpi` controls how sharp the file comes out: 150 is enough to project, 300 for print.
`bbox_inches="tight"` trims the spare white margin.

`plt.close(fig)` closes the figure when you are done. A loop that draws fifty and closes none
keeps all fifty in memory, and matplotlib eventually warns you about it.

## What that chart is missing

The one above is technically correct and says nothing. It has no title, the axis numbers are
unlabelled, and the reader has to guess what 1 to 12 means.

| Element | What it adds | Method |
|---|---|---|
| Title | The finding, in one sentence | `set_title` |
| Axis label | What is measured, and in what unit | `set_ylabel` |
| Zero baseline | That the difference is not exaggerated | `set_ylim` |
| Source | Where the numbers came from | `fig.text` |

The same data, told properly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(MONTHS, monthly.values / 1000, marker="o", linewidth=2, color="#2B5F8F")

ax.set_title("Revenue by month, 2025", fontsize=14, fontweight="bold", loc="left")
ax.set_ylabel("Thousands of pesos")
ax.set_ylim(bottom=0)          # a bar or a line starts at zero, or it lies

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.show()
plt.close(fig)

Everything stripped from the frame was ink that was not saying anything.

## The cut axis, which is how you lie with true numbers

`set_ylim(bottom=0)` is not decoration. Cutting the axis exaggerates the difference, and doing
it on purpose is the most common way to lie with a chart that contains only correct numbers.

In [ ]:
# DRAWS BADLY ON PURPOSE, on the left. The same four numbers in both.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(by_region.index, by_region.values, color="#B4530A")
axes[0].set_ylim(1400, 4500)                      # the cut axis
axes[0].set_title("Wrong: South looks like nothing", loc="left", fontweight="bold")

axes[1].bar(by_region.index, by_region.values, color="#2B5F8F")
axes[1].set_ylim(bottom=0)
axes[1].set_title("Right: South sells a third of North", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Thousands of pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("North against South:", round(by_region["North"] / by_region["South"], 2), "times")

North sells 2.8 times what South does. On the left chart it looks like twenty times. The four
numbers are the same and none of them is wrong.

## Several charts at once

`subplots` takes a grid. The axes come back as an array you index into.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

by_region_sorted = sales.groupby("region")["amount"].sum().sort_values() / 1000
by_channel = sales.groupby("channel")["amount"].sum().sort_values() / 1000

axes[0].barh(by_region_sorted.index, by_region_sorted.values, color="#3776AB")
axes[0].set_title("By region", loc="left", fontweight="bold")

axes[1].barh(by_channel.index, by_channel.values, color="#3776AB")
axes[1].set_title("By channel", loc="left", fontweight="bold")

for ax in axes:
    ax.set_xlabel("Thousands of pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()          # keeps one chart's labels off the other
plt.show()
plt.close(fig)

---
# Block 3 · Making it readable without you

The chart is going to travel alone in an email. Anything you would have to explain out loud is
something it is missing in writing.

## The title states the finding

"Revenue by month" describes the axes, which the reader can already see. "December carried
20 % of the year's revenue" is what you actually found.

A chart titled with its conclusion is read once. A chart titled with its axes is stared at
until somebody explains it.

In [ ]:
peak = monthly.idxmax()
share = monthly.max() / monthly.sum()

print(f"The peak month is {peak} and it took {share:.1%} of the year")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(MONTHS, monthly.values, color="#C7D6E8", edgecolor="none")

# One bar carries the point, so one bar gets the strong colour.
bars[peak - 1].set_color("#2B5F8F")

ax.set_title(f"December carried {share:.0%} of the year's revenue",
             fontsize=15, fontweight="bold", loc="left", pad=18)

# The subtitle is where the description goes, now that the title says the point.
ax.text(0, 1.02, "Revenue by month, 2025", transform=ax.transAxes,
        fontsize=10.5, color="#5B6B84")

# 2567118.5 makes the reader count digits. 2.6M reads without thinking.
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / 1_000_000:.1f}M"))
ax.set_ylabel("Revenue")
ax.set_ylim(bottom=0)

ax.annotate(f"{monthly.max() / 1_000_000:.2f}M",
            xy=(peak - 1, monthly.max()), xytext=(0, 8), textcoords="offset points",
            ha="center", fontweight="bold", color="#2B5F8F")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="y", length=0)

# A chart with no source is an opinion.
fig.text(0.125, -0.02, "Source: sales_clean.csv, 306 records, 2025",
         fontsize=9, color="#5B6B84")

plt.show()
plt.close(fig)

Four things changed from the previous version, and none of them touched the data.

The **title** states the finding and the subtitle kept the description. A **single bar**
carries the strong colour: if everything is emphasised, nothing is, and the other eleven are
still there and still comparable, they have only stopped competing for attention. The
**formatter** changes the axis labels without touching the underlying values. And the
**source** at the foot turns an opinion into evidence.

## Colour that survives greyscale

Around one man in twelve has some form of colour blindness, and every chart eventually gets
printed in black and white. Two defences:

1. **Use a palette designed for it.** Blue against orange separates for almost everyone; red
   against green does not.
2. **Do not let colour be the only signal.** Line style, marker shape and a direct label all
   survive being turned grey.

In [ ]:
by_channel_month = sales.pivot_table(index=sales["date"].dt.month,
                                     columns="channel", values="amount", aggfunc="sum")

SAFE = {"Retail": "#2B5F8F", "Online": "#B4530A", "Wholesale": "#5B6B84"}
STYLE = {"Retail": "-", "Online": "--", "Wholesale": ":"}
MARKER = {"Retail": "o", "Online": "s", "Wholesale": "^"}

fig, ax = plt.subplots(figsize=(10, 5))

for channel in by_channel_month.columns:
    ax.plot(MONTHS, by_channel_month[channel] / 1000, label=channel, color=SAFE[channel],
            linestyle=STYLE[channel], marker=MARKER[channel], linewidth=2)

    # A label at the end of the line beats a legend: the reader's eye never has to
    # leave the data to find out which line is which.
    ax.annotate(channel, xy=(11, by_channel_month[channel].iloc[-1] / 1000),
                xytext=(8, 0), textcoords="offset points",
                color=SAFE[channel], fontweight="bold", va="center")

ax.set_title("Wholesale drives the December peak",
             fontsize=15, fontweight="bold", loc="left", pad=18)
ax.text(0, 1.02, "Revenue by channel and month, thousands of pesos",
        transform=ax.transAxes, fontsize=10.5, color="#5B6B84")
ax.set_ylim(bottom=0)
ax.set_xlim(-0.4, 12.6)          # room on the right for the end labels
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.show()
plt.close(fig)

That chart still reads printed in grey, because every line carries three signals besides
colour: its stroke style, its marker and its name written at the end.

## The alternative text

A chart in a report or on a web page needs a written description for anyone using a screen
reader. Write it as the sentence you would say out loud if the image failed to load: what it
shows, and what it shows you.

And write it against the table, not from memory. Describing a trend the data does not have is
the easiest way to make an accessible chart say something false.

In [ ]:
print((by_channel_month / 1000).round(0).to_string())

In [ ]:
alt_text = (
    "Line chart of 2025 revenue by month for three sales channels, in thousands of "
    "pesos. Retail stays between 120 and 320 all year. Online swings between 36 and 656 "
    "with no clear trend. Wholesale is the largest channel in ten of the twelve months "
    "and jumps from 322 in November to 1,611 in December, which is what produces the "
    "year-end peak."
)
print(alt_text)

Every figure in that paragraph can be checked against the table above, and that is why it can
be written without worry. Check it yourself.

In [ ]:
table = (by_channel_month / 1000).round(0)

print("Retail runs from", table["Retail"].min(), "to", table["Retail"].max())
print("Online runs from", table["Online"].min(), "to", table["Online"].max())
print("Months where Wholesale is largest:",
      (table.idxmax(axis=1) == "Wholesale").sum(), "of 12")
print("Wholesale in November:", table["Wholesale"].iloc[10],
      "| in December:", table["Wholesale"].iloc[11])

---
## Four ways to ruin a correct chart

**Cutting the vertical axis.** A two per cent difference looks like fifty. The numbers are
right and the chart lies. You saw it with North against South.

**Unsorted bars.** The reader has to do the ranking by eye. Sorting is free and answers the
question on its own.

**A line across categories.** Connecting North to South suggests a journey that is not there.
Categories get bars.

**Leaving the default title.** A chart with no title and no source is an opinion. With both it
is evidence.

---
# Exercises

The solutions sit at the very bottom.

### Exercise 1 · Choosing without drawing

For each question, say in a comment which chart you would use and why. Do not draw anything
yet.

1. Which of the three channels sells most?
2. Did revenue grow or fall through the year?
3. Are the months with more sales also the ones with the highest average ticket?
4. How even is the size of the sales?

### Exercise 2 · All four, with the course data

Draw one of each type using the course tables: a bar, a line, a scatter and a histogram. Give
all of them a title, an axis label and a zero baseline where it applies.

Use a two by two grid, with `plt.subplots(2, 2)`.

### Exercise 3 · From description to finding

Take the revenue-by-region chart and write three different titles for it:

1. One that describes the axes.
2. One that states the finding with a figure.
3. One that states the finding with a comparison.

Draw the third version in full, with subtitle, formatted axes and source.

### Exercise 4 · The histogram of the sales

Make a histogram of the `amount` column of `sales`. Draw the mean and the median on top, in
different colours and styles, and label both.

Then answer in a comment which of the two better describes a typical sale, and why they sit so
far apart.

### Exercise 5 · The same figure, honest and crooked

Take revenue by channel and draw two versions side by side: one with the axis starting at
zero, and one with the axis cut so the difference looks enormous.

Work out and print the real ratio between the biggest and smallest channel, so it is clear how
much the second one exaggerates.

### Exercise 6 · Verifiable alternative text

Write the alternative text for the chart from exercise 3. Then write the code that checks
every figure you mentioned, the way it was done above.

If any figure cannot be checked with one line of pandas, take it out of the text.

### Exercise 7 · A chart from your project, finished

Produce a chart with your project data: a title that states the finding, a descriptive
subtitle, formatted axes, one highlighted element and the source at the foot. Write its
alternative text too.

No pie charts, and the vertical axis starts at zero.

The test: show it without saying anything. If your classmate asks what it shows, the title is
missing the finding.

---
## Three ideas to take away

**The question picks the chart.** Bar compares, line changes over time, scatter relates and
histogram spreads. Picking the shape first and finding data for it afterwards is how pretty
charts that say nothing get made.

**Title with the finding.** The names of the axes are already visible. What the reader cannot
see on their own is what you found.

**Colour never travels alone.** Line style, marker or a direct label. All of those survive a
greyscale print and a reader who cannot tell two of your colours apart.

Next session is seaborn, which does in one line several of the things that took eight today,
plus the close of the integrating project.

---
# Solutions

### Exercise 1

```python
# 1. Bars. Three named categories and the question is how they compare.
#    Sorted, so the order answers on its own.
# 2. Line. The axis is time and the months sit in an order that means something.
# 3. Scatter. Two figures per month and the question is whether they move
#    together. One dot per month, sales on one axis and average ticket on the other.
# 4. Histogram. It is a single column and the question is how it is spread out,
#    not how it compares against something else.
```

The fourth is the one people get wrong. "How even" sounds like a comparison and it is not:
there is one variable, and what you want to see is its shape.

### Exercise 2

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

by_channel = sales.groupby("channel")["amount"].sum().sort_values() / 1000
axes[0, 0].barh(by_channel.index, by_channel.values, color="#2B5F8F")
axes[0, 0].set_title("Wholesale brings half the revenue", loc="left", fontweight="bold")
axes[0, 0].set_xlabel("Thousands of pesos")

axes[0, 1].plot(MONTHS, monthly.values / 1000, marker="o", color="#2B5F8F", linewidth=2)
axes[0, 1].set_title("December breaks the scale", loc="left", fontweight="bold")
axes[0, 1].set_ylabel("Thousands of pesos")
axes[0, 1].set_ylim(bottom=0)

axes[1, 0].scatter(sales["units"], sales["amount"] / 1000,
                   alpha=0.5, color="#2B5F8F", edgecolor="none")
axes[1, 0].set_title("More units is not always more money", loc="left", fontweight="bold")
axes[1, 0].set_xlabel("Units")
axes[1, 0].set_ylabel("Thousands of pesos")

axes[1, 1].hist(sales["units"], bins=20, color="#2B5F8F", edgecolor="white")
axes[1, 1].set_title("Most sales are small", loc="left", fontweight="bold")
axes[1, 1].set_xlabel("Units")
axes[1, 1].set_ylabel("Sales")

for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()
```

The scatter at the bottom left is the interesting one: horizontal bands appear, one per
product, because the amount is units times a price that only takes five values. A chart can
show you the structure of the file as well as the answer you were after.

### Exercise 3

```python
# 1. "Revenue by region"                        describes the axes
# 2. "North carried 34 % of revenue"            finding with a figure
# 3. "North sells almost three times South"     finding with a comparison

north_share = by_region["North"] / by_region.sum()
times = by_region["North"] / by_region["South"]

ordered = by_region.sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(ordered.index, ordered.values, color="#C7D6E8")
bars[-1].set_color("#2B5F8F")

ax.set_title(f"North sells {times:.1f} times what South sells",
             fontsize=15, fontweight="bold", loc="left", pad=18)
ax.text(0, 1.04, "Revenue by region, 2025", transform=ax.transAxes,
        fontsize=10.5, color="#5B6B84")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / 1000:.1f}M"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.text(0.125, -0.03, "Source: sales_clean.csv, 306 records, 2025",
         fontsize=9, color="#5B6B84")
plt.show()
```

The third is the most useful of the three because it does not require the reader to know
whether 34 % is a lot. A comparison brings its own reference.

### Exercise 4

```python
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(sales["amount"] / 1000, bins=30, color="#2B5F8F", edgecolor="white")

mean_sale = sales["amount"].mean() / 1000
median_sale = sales["amount"].median() / 1000

ax.axvline(mean_sale, color="#B4530A", linestyle="--", linewidth=2)
ax.axvline(median_sale, color="#0B1B3A", linestyle=":", linewidth=2)
ax.annotate(f"mean {mean_sale:,.0f}k", xy=(mean_sale, 0), xytext=(6, 40),
            textcoords="offset points", color="#B4530A", fontweight="bold")
ax.annotate(f"median {median_sale:,.0f}k", xy=(median_sale, 0), xytext=(-90, 60),
            textcoords="offset points", color="#0B1B3A", fontweight="bold")

ax.set_title("The typical sale is much smaller than the mean",
             loc="left", fontweight="bold")
ax.set_xlabel("Thousands of pesos per sale")
ax.set_ylabel("Sales")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

# The median describes a typical sale better. The spread has a long tail to the
# right: a few espresso machine sales are worth twenty times what a mug sale is,
# and those pull the mean up without most of the data going anywhere near it.
```

This is why a serious report gives the mean and the median together. When they separate this
much, the separation is the finding.

### Exercise 5

```python
channel = sales.groupby("channel")["amount"].sum() / 1000

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(channel.index, channel.values, color="#2B5F8F")
axes[0].set_ylim(bottom=0)
axes[0].set_title("Honest", loc="left", fontweight="bold")

axes[1].bar(channel.index, channel.values, color="#B4530A")
axes[1].set_ylim(channel.min() * 0.97, channel.max() * 1.01)
axes[1].set_title("Crooked: the axis starts near the minimum", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Thousands of pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("Real ratio, largest to smallest:", round(channel.max() / channel.min(), 2))
```

The real ratio is close to four and the crooked version makes it look like twenty. Worth
drawing once, because it is the manipulation you will meet most often in other people's
charts.

### Exercise 6

```python
text = (
    "Horizontal bar chart of 2025 revenue by region, in millions of pesos. North is "
    "the tallest at 4.35 million, followed by Centre at 3.92 and West at 3.03. South "
    "is the lowest at 1.55 million, roughly a third of North."
)
print(text)

print("North:", round(by_region['North'] / 1000, 2), "million")
print("Centre:", round(by_region['Centre'] / 1000, 2))
print("West:", round(by_region['West'] / 1000, 2))
print("South:", round(by_region['South'] / 1000, 2))
print("South as a share of North:", round(by_region['South'] / by_region['North'], 2))
```

Note that the description gives the order and the figures, not adjectives. "North clearly
dominates" is no use to somebody who cannot see the chart; "4.35 against 1.55 million" is.

### Exercise 7

There is no published solution, because the data is different for everyone. It is graded on
five things: a title with the finding, a descriptive subtitle, formatted axes, one highlighted
element and the source at the foot. The alternative text is graded separately, and every
figure it mentions has to be checkable.